# 🌍 旅游助手 Agent

智能旅游助手，支持目的地查询、行程规划、住宿美食推荐等功能

In [ ]:
import os
import sys
from pathlib import Path

# 添加项目路径
sys.path.insert(0, str(Path.cwd()))

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.memory import ConversationBufferWindowMemory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from rich import print as rprint
import dotenv

dotenv.load_dotenv(override=True)

# 初始化 LLM
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
    temperature=0.7
)

print("LLM 初始化完成")

## 1. 导入工具

In [ ]:
from tools import (
    get_weather,
    search_places,
    get_place_details,
    calculate,
    estimate_budget,
    search_travel_knowledge,
)

# 工具列表
tools = [
    get_weather,
    search_places,
    get_place_details,
    calculate,
    estimate_budget,
    search_travel_knowledge,
]

rprint(f"[green]✓ 已加载 {len(tools)} 个工具[/green]")
for tool in tools:
    rprint(f"  • {tool.name}: {tool.description[:30]}...")

## 2. 创建 Agent

In [ ]:
from prompts import SYSTEM_PROMPT

# 创建 Agent
agent = create_agent(llm, tools)

# 创建记忆
memory = ConversationBufferWindowMemory(
    k=10,
    return_messages=True,
    memory_key="history",
)

rprint("[green]✓ Agent 创建成功[/green]")

## 3. 对话函数

In [ ]:
def chat(user_input: str, stream: bool = False):
    """与旅游助手对话"""
    # 加载记忆
    history = memory.load_memory_variables({})
    
    # 构建消息
    messages = [("system", SYSTEM_PROMPT)]
    
    # 添加历史消息
    for msg in history.get("history", []):
        if isinstance(msg, HumanMessage):
            messages.append(("human", msg.content))
        elif isinstance(msg, AIMessage):
            messages.append(("ai", msg.content))
    
    messages.append(("human", user_input))
    
    if stream:
        # 流式输出
        full_response = ""
        for chunk in agent.stream({"messages": messages}):
            if "agent" in chunk:
                for msg in chunk["agent"]["messages"]:
                    if msg.content:
                        print(msg.content, end="", flush=True)
                        full_response += msg.content
        print()
        response = full_response
    else:
        # 非流式输出
        result = agent.invoke({"messages": messages})
        response = result["messages"][-1].content
    
    # 保存到记忆
    memory.save_context(
        {"input": user_input},
        {"output": response}
    )
    
    return response

## 4. 测试天气查询

In [ ]:
# 测试天气查询
rprint("[bold]测试天气查询[/bold]\n")
response = chat("北京今天天气怎么样？")
rprint(f"[green]小旅:[/green] {response}")

## 5. 测试景点查询

In [ ]:
# 测试景点查询
rprint("[bold]测试景点查询[/bold]\n")
response = chat("西安有什么好玩的景点？")
rprint(f"[green]小旅:[/green] {response}")

## 6. 测试行程规划

In [ ]:
# 测试行程规划
rprint("[bold]测试行程规划[/bold]\n")
response = chat("我想去成都玩3天，帮我规划一下行程")
rprint(f"[green]小旅:[/green] {response}")

## 7. 测试预算估算

In [ ]:
# 测试预算估算
rprint("[bold]测试预算估算[/bold]\n")
response = chat("去杭州旅游3天大概需要多少钱？")
rprint(f"[green]小旅:[/green] {response}")

## 8. 测试知识查询

In [ ]:
# 测试知识查询
rprint("[bold]测试知识查询[/bold]\n")
response = chat("出国旅游需要准备什么行李？")
rprint(f"[green]小旅:[/green] {response}")

## 9. 测试多轮对话

In [ ]:
# 测试多轮对话
rprint("[bold]测试多轮对话[/bold]\n")

# 第一轮
rprint("[cyan]用户:[/cyan] 我想去北京旅游")
response1 = chat("我想去北京旅游")
rprint(f"[green]小旅:[/green] {response1[:200]}...\n")

# 第二轮
rprint("[cyan]用户:[/cyan] 3天，2个人，预算5000")
response2 = chat("3天，2个人，预算5000")
rprint(f"[green]小旅:[/green] {response2[:200]}...")

## 10. 流式输出示例

In [ ]:
# 流式输出
rprint("[bold]流式输出测试[/bold]\n")
rprint("[cyan]用户:[/cyan] 推荐几个上海的美食")
rprint("[green]小旅:[/green] ", end="")
chat("推荐几个上海的美食", stream=True)

## 11. 交互式对话

In [ ]:
# 交互式对话（取消注释运行）

# from rich.console import Console
# console = Console()
# 
# while True:
#     user_input = console.input("[bold cyan]你: [/bold cyan]")
#     if user_input.lower() in ["quit", "exit"]:
#         print("再见！")
#         break
#     if user_input.lower() == "clear":
#         memory.clear()
#         print("记忆已清空")
#         continue
#     if not user_input.strip():
#         continue
#     
#     console.print("[bold green]小旅: [/bold green]", end="")
#     chat(user_input, stream=True)
#     print()

## 总结

### 功能列表
| 功能 | 工具 | 示例 |
|------|------|------|
| 天气查询 | `get_weather` | 北京天气怎么样？ |
| 景点搜索 | `search_places` | 西安有什么景点？ |
| 景点详情 | `get_place_details` | 故宫详细信息 |
| 预算估算 | `estimate_budget` | 3天需要多少钱？ |
| 旅游知识 | `search_travel_knowledge` | 出国带什么行李？ |
| 数学计算 | `calculate` | 汇率换算 |

### 使用方式
1. **Notebook**: 运行上方代码块
2. **命令行**: `python main.py`